# V2 calendar cGAN research

I use this notebook as a thin interface to `python -m cgan_research`. I select a GPU runtime, inspect it, and choose persistent storage before a measured pilot. Opening or running the notebook with its default switches starts no research training. The standalone CPU smoke cell creates **synthetic demonstration data**, runs two optimizer steps, and checks the complete package workflow; it establishes no measured-data performance or GPU support.

I retain Colab’s installed CUDA PyTorch. I do not run the repository’s CPU installer. I keep household identifiers, raw data, and checkpoints outside Git. Weather and EV/PV/heat-pump conditions are excluded from this calendar baseline. The local tested Python/PyTorch versions are recorded in `docs/colab.md`; actual Colab versions are recorded below.

In [ ]:
from pathlib import Path
import os
import sys
import json
import subprocess
import platform
from datetime import datetime, timezone

REPO = Path(os.environ.get('CGAN_REPO', '/content/CGAN_Research')).resolve()
REPOSITORY_URL = 'https://github.com/MahmoudAsadi97/CGAN_Research.git'
EXPECTED_COMMIT = ''  # I may pin a reviewed V2 commit after checkout.
WORK_ROOT = None  # I set an explicit Path, preferably inside my selected Drive folder.
RUN_NAME = 'calendar-pilot-seed17'
RESUME_CHECKPOINT = None  # Explicit path to this run's last.pt; same data/config required.
RUN_CHECKOUT = False
RUN_INSTALL = False
MOUNT_DRIVE = False
RUN_DOWNLOAD = False
RUN_PREPARE = False
RUN_PILOT = False
RUN_RESUME = False
RUN_EVALUATE = False
RUN_PLOT = False
RUN_CPU_SMOKE = False
os.environ.setdefault('MPLBACKEND', 'Agg')
os.environ.setdefault('MPLCONFIGDIR', '/tmp/cgan-mpl')
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
print('Python:', sys.version)
print('Repository:', REPO)
print('All research actions require their explicit switch.')

## Checkout and compatible dependencies

I enable checkout once, inspect the exact V2 SHA, and optionally pin it. I do not reset local changes. Installation keeps the existing Torch package; a missing or incompatible CUDA installation needs an explicit runtime repair before the GPU pilot.

In [ ]:
if globals().get('RUN_CHECKOUT', False):
    if not REPO.exists():
        subprocess.run(['git', 'clone', '--branch', 'V2', '--single-branch', REPOSITORY_URL, str(REPO)], check=True)
    dirty = subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip()
    if dirty:
        raise RuntimeError('I inspect local changes before switching checkout.')
    subprocess.run(['git', '-C', str(REPO), 'checkout', 'V2'], check=True)
    if EXPECTED_COMMIT:
        subprocess.run(['git', '-C', str(REPO), 'merge-base', '--is-ancestor', EXPECTED_COMMIT, 'V2'], check=True)
        subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', EXPECTED_COMMIT], check=True)
    print(subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True))

In [ ]:
if globals().get('RUN_INSTALL', False):
    import torch
    original_torch = (str(torch.__version__), torch.version.cuda)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', str(REPO)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'numpy>=1.24', 'pandas>=2.1', 'scipy>=1.10', 'matplotlib>=3.7', 'PyYAML>=6', 'requests>=2.31', 'nbformat>=5'], check=True)
    installed_torch = subprocess.check_output([sys.executable, '-c', 'import torch,json;print(json.dumps([str(torch.__version__),torch.version.cuda]))'], text=True)
    if tuple(json.loads(installed_torch)) != original_torch:
        raise RuntimeError('Torch changed; I inspect the runtime before proceeding.')
    print('Retained Torch:', original_torch)

In [ ]:
# This read-only inspection runs after checkout/install, or against an existing local checkout.
import torch
environment = {'recorded_utc': datetime.now(timezone.utc).isoformat(), 'python': platform.python_version(),
               'torch': str(torch.__version__), 'cuda_build': torch.version.cuda,
               'cuda_available': torch.cuda.is_available(), 'colab_gpu_verified': False}
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    environment.update(gpu=torch.cuda.get_device_name(0), gpu_total_bytes=total, gpu_free_bytes=free)
if (REPO / '.git').exists():
    environment['git_sha'] = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
    environment['git_dirty'] = bool(subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip())
print(json.dumps(environment, indent=2))
if not environment['cuda_available']:
    print('GPU research path unavailable. The separately selected CPU synthetic smoke remains available.')

## My persistent workspace

I explicitly authenticate Drive, then set `WORK_ROOT = Path('/content/drive/MyDrive/<my chosen research folder>')` in the settings cell. Paths with spaces are supported. Checkpoints and command logs are written directly under that location. For an ephemeral trial I may deliberately choose `/content/...`; those artifacts disappear with the runtime. I verify a saved checkpoint after a new runtime before claiming restart persistence.

In [ ]:
if globals().get('MOUNT_DRIVE', False):
    from google.colab import drive
    drive.mount('/content/drive')
    print('I now select my own WORK_ROOT path in the settings cell.')

In [ ]:
def workspace():
    if WORK_ROOT is None:
        raise ValueError('I must select WORK_ROOT before a research action.')
    root = Path(WORK_ROOT).expanduser().resolve()
    if str(root).startswith('/content/drive/') and not Path('/content/drive/MyDrive').is_dir():
        raise RuntimeError('I mount Drive before creating persistent artifact paths.')
    root.mkdir(parents=True, exist_ok=True)
    (root / 'environment.json').write_text(json.dumps(environment, indent=2) + '\n')
    return root

def cli(*arguments):
    root = workspace()
    env = dict(os.environ, PYTHONPATH=str(REPO / 'src') + os.pathsep + os.environ.get('PYTHONPATH', ''))
    command = [sys.executable, '-m', 'cgan_research', *map(str, arguments)]
    started = datetime.now(timezone.utc).isoformat()
    completed = subprocess.run(command, cwd=REPO, env=env, capture_output=True, text=True)
    record = {'started_utc': started, 'command': command, 'returncode': completed.returncode,
              'stdout': completed.stdout, 'stderr': completed.stderr}
    with (root / 'commands.jsonl').open('a') as stream:
        stream.write(json.dumps(record) + '\n')
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr, file=sys.stderr)
    completed.check_returncode()
    return completed


## Audited electricity preparation

I use the verified release URL and checksum from `configs/dataset_electricity.yaml`. Audit must pass before preparation. The adapter preserves negative net energy, sums quarter-hour kWh into hourly kWh, and quarantines ambiguous DST/incomplete days. The grouping is a meter proxy, as recorded in the dataset audit. No weather join is performed.

In [ ]:
if globals().get('RUN_DOWNLOAD', False):
    import yaml
    root = workspace()
    dataset_config_path = REPO / 'configs' / 'dataset_electricity.yaml'
    source = yaml.safe_load(dataset_config_path.read_text())['source']
    cli('download', '--url', source['url'], '--sha256', source['sha256'], '--output', root / 'raw.zip')

In [ ]:
if globals().get('RUN_PREPARE', False):
    root = workspace()
    dataset_config_path = REPO / 'configs' / 'dataset_electricity.yaml'
    cli('audit', '--kind', 'electricity', '--input', root / 'raw.zip', '--config', dataset_config_path, '--output', root / 'audit.json')
    cli('prepare', '--kind', 'electricity', '--input', root / 'raw.zip', '--config', dataset_config_path, '--output', root / 'prepared')
    manifest = json.loads((root / 'prepared' / 'manifest.json').read_text())
    print('Data fingerprint:', manifest['data_fingerprint'])
    print('Split fingerprint:', manifest['split_fingerprint'])
    print('Partition counts:', manifest['counts'])

## Bounded GPU pilot or explicit resume

I inspect available GPU memory and the baseline bounds before enabling one action. The default pilot ends at the first limit: 400 optimizer steps, 600 cumulative seconds checked between batches, or 10,000 epochs. These are pilot bounds, not a full research run. I record measured throughput/memory and freeze the complete experiment budget before any full multi-seed experiment. Resume keeps the same run path, configuration, and data fingerprints; reaching the original budget does not extend it.

In [ ]:
if globals().get('RUN_PILOT', False) or globals().get('RUN_RESUME', False):
    if RUN_PILOT and RUN_RESUME:
        raise ValueError('I select either a new pilot or resume.')
    if not torch.cuda.is_available():
        raise RuntimeError('The research pilot requires an available CUDA GPU.')
    root = workspace()
    config_path = REPO / 'configs' / 'calendar_baseline.yaml'
    import yaml
    config = yaml.safe_load(config_path.read_text())['training']
    if config['max_steps'] > 400 or config['max_seconds'] > 600:
        raise ValueError('This pilot cell only accepts the declared bounded pilot.')
    arguments = ['train', '--prepared', root / 'prepared', '--config', config_path, '--output', root / RUN_NAME]
    if RUN_RESUME:
        if RESUME_CHECKPOINT is None:
            raise ValueError('I select an explicit RESUME_CHECKPOINT.')
        arguments += ['--resume', Path(RESUME_CHECKPOINT)]
    cli(*arguments)
    print((root / RUN_NAME / 'status.json').read_text())

## Validation evaluation and reproducible figures

I choose the validation-selected checkpoint when available. A last checkpoint fallback is explicitly provisional. This cell evaluates validation only; opening the frozen test requires a separate deliberate CLI invocation with `--confirm-test`. Every generated sample is retained. I use the same frozen figure scale YAML for all model comparisons; without that file the manifest records evaluation-derived scales.

In [ ]:
FIGURE_CONFIG = None  # I select one shared YAML with fixed limits before comparing candidate models.
if globals().get('RUN_EVALUATE', False):
    root = workspace()
    checkpoint = root / RUN_NAME / 'best.pt'
    if not checkpoint.exists():
        checkpoint = root / RUN_NAME / 'last.pt'
        print('Provisional pilot: no selected best checkpoint available; using last.pt.')
    cli('generate', '--checkpoint', checkpoint, '--samples-per-condition', '2000', '--seed', '42', '--output', root / 'generated.npz')
    cli('evaluate', '--prepared', root / 'prepared', '--generated', root / 'generated.npz', '--split', 'validation', '--output', root / 'evaluation')

In [ ]:
if globals().get('RUN_PLOT', False):
    root = workspace()
    arguments = ['plot', '--evaluation', root / 'evaluation', '--output', root / 'figures']
    if FIGURE_CONFIG is not None:
        arguments += ['--config', Path(FIGURE_CONFIG)]
    cli(*arguments)
    figure_manifest = json.loads((root / 'figures' / 'figure_manifest.json').read_text())
    print('Exported families:', [item['family'] for item in figure_manifest['figures']])
    print('Warnings:', figure_manifest['warnings'])

## Standalone CPU synthetic smoke

I run this cell independently after making the package available. I set `RUN_CPU_SMOKE = True` first. It is self-contained: `CGAN_REPO` can select an existing checkout and `CGAN_SMOKE_DIR` can select a new output path. It does not require Drive, CUDA, downloads, or the research settings/helpers. It creates a fresh synthetic fixture, audits/prepares it, trains two CPU steps, generates eight samples, evaluates validation, and exports figures through the package CLI. Existing output paths are refused.

In [ ]:
if globals().get('RUN_CPU_SMOKE', False):
    import os
    import sys
    import json
    import tempfile
    import subprocess
    from pathlib import Path
    import numpy as np
    import pandas as pd
    import yaml

    smoke_repo = Path(os.environ.get('CGAN_REPO', globals().get('REPO', Path.cwd()))).resolve()
    selected_output = os.environ.get('CGAN_SMOKE_DIR')
    smoke_root = Path(selected_output) if selected_output else Path(tempfile.mkdtemp(prefix='cgan-synthetic-')) / 'smoke'
    smoke_root.mkdir(parents=True, exist_ok=False)
    rng = np.random.default_rng(17)
    records = []
    for household in range(40):
        for day in ('2024-01-01', '2024-01-06'):
            profile = 1 + np.sin(np.arange(24) * np.pi / 12) + rng.normal(0, .2, 24)
            records.append({'profile_id': f'synthetic-{household}-{day}', 'household_id': f'synthetic-{household}',
                            'date': day, **{f'load_{hour:02}': float(value) for hour, value in enumerate(profile)}})
    pd.DataFrame(records).to_csv(smoke_root / 'synthetic.csv', index=False)
    (smoke_root / 'dataset.yaml').write_text(yaml.safe_dump({'dataset': {'synthetic': True, 'timezone': 'Europe/Brussels', 'units': 'kWh', 'interval_convention': 'start'}}))
    training = {'device': 'cpu', 'max_steps': 2, 'max_epochs': 2, 'batch_size': 16, 'max_seconds': 60,
                'checkpoint_interval': 1, 'validation_interval': 1, 'samples_per_condition': 4, 'seed': 17}
    (smoke_root / 'pilot.yaml').write_text(yaml.safe_dump({'training': training}))
    smoke_env = dict(os.environ, PYTHONPATH=str(smoke_repo / 'src') + os.pathsep + os.environ.get('PYTHONPATH', ''),
                     CUDA_VISIBLE_DEVICES='', MPLBACKEND='Agg', MPLCONFIGDIR='/tmp/cgan-mpl', OPENBLAS_NUM_THREADS='1')
    commands = [
        ['audit', '--kind', 'daily', '--input', smoke_root / 'synthetic.csv', '--config', smoke_root / 'dataset.yaml', '--output', smoke_root / 'audit.json'],
        ['prepare', '--kind', 'daily', '--input', smoke_root / 'synthetic.csv', '--config', smoke_root / 'dataset.yaml', '--output', smoke_root / 'prepared'],
        ['train', '--prepared', smoke_root / 'prepared', '--config', smoke_root / 'pilot.yaml', '--output', smoke_root / 'run'],
        ['generate', '--checkpoint', smoke_root / 'run' / 'last.pt', '--samples-per-condition', '4', '--seed', '42', '--output', smoke_root / 'generated.npz'],
        ['evaluate', '--prepared', smoke_root / 'prepared', '--generated', smoke_root / 'generated.npz', '--split', 'validation', '--reference-replicates', '2', '--output', smoke_root / 'evaluation'],
        ['plot', '--evaluation', smoke_root / 'evaluation', '--output', smoke_root / 'figures']]
    with (smoke_root / 'commands.jsonl').open('w') as log:
        for arguments in commands:
            command = [sys.executable, '-m', 'cgan_research', *map(str, arguments)]
            completed = subprocess.run(command, cwd=smoke_repo, env=smoke_env, capture_output=True, text=True)
            log.write(json.dumps({'command': command, 'returncode': completed.returncode, 'stdout': completed.stdout, 'stderr': completed.stderr}) + '\n')
            if completed.returncode:
                raise RuntimeError(completed.stdout + completed.stderr)
    smoke_evaluation = json.loads((smoke_root / 'evaluation' / 'evaluation.json').read_text())
    smoke_figures = json.loads((smoke_root / 'figures' / 'figure_manifest.json').read_text())
    assert smoke_evaluation['metadata']['provenance']['synthetic'] is True
    assert smoke_evaluation['metadata']['n_generated_total'] == 8
    assert len(smoke_evaluation['rows']) == 80
    assert smoke_figures['synthetic'] is True and len(smoke_figures['figures']) == 8
    print('SYNTHETIC CPU SMOKE ONLY:', smoke_root)
    print('No measured-data result, Colab GPU, or Drive restart persistence is established by this cell.')